#**MovieLens 1M Analysis: Colaborative Filtering**  
**Group L03 - Team 07**

# 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import pickle
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
# # if running on gg colab
# import gdown
# gdown.download_folder('https://drive.google.com/drive/folders/1HkzgKNjgJ6H9edInSBvFj60gc52y4equ?usp=drive_link', quiet=False, use_cookies=False, output="./")

Retrieving folder contents


Processing file 1C_MwVj5tpshYnbHuDTNoYPN6m0lO0CV0 movies.dat
Processing file 1RaGo4q8ofqSaVYOExl70ARP5BvdwAK-Z ratings.dat
Processing file 1AH9EDZHQKyRvzOS74YbzdhNXPFa7lLaz README
Processing file 1XPeMfuH0dxJIy2zIk7s6n1QZ5iVWPgEy users.dat


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1C_MwVj5tpshYnbHuDTNoYPN6m0lO0CV0
To: /content/ml-1m/movies.dat
100%|██████████| 175k/175k [00:00<00:00, 50.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1RaGo4q8ofqSaVYOExl70ARP5BvdwAK-Z
To: /content/ml-1m/ratings.dat
100%|██████████| 25.6M/25.6M [00:00<00:00, 45.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1AH9EDZHQKyRvzOS74YbzdhNXPFa7lLaz
To: /content/ml-1m/README
100%|██████████| 5.75k/5.75k [00:00<00:00, 14.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1XPeMfuH0dxJIy2zIk7s6n1QZ5iVWPgEy
To: /content/ml-1m/users.dat
100%|██████████| 140k/140k [00:00<00:00, 72.8MB/s]
Download completed


['./ml-1m/movies.dat',
 './ml-1m/ratings.dat',
 './ml-1m/README',
 './ml-1m/users.dat']

In [3]:
DATA_PATH = './ml-1m/'
ratings = pd.read_csv(DATA_PATH + 'ratings.dat', sep='::', engine='python', names=['UserID', 'MovieID', 'Rating', 'Timestamp'])
users = pd.read_csv(DATA_PATH + 'users.dat', sep='::', engine='python', names=['UserID', 'Gender', 'Age', 'Occupation', 'Zipcode'])
movies = pd.read_csv(DATA_PATH + 'movies.dat', sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin-1')
print(f"Ratings: {len(ratings):,} | Users: {len(users):,} | Movies: {len(movies):,}")

Ratings: 1,000,209 | Users: 6,040 | Movies: 3,883


In [4]:
print(users[users['UserID'] == 1])

   UserID Gender  Age  Occupation Zipcode
0       1      F    1          10   48067


In [5]:
df = ratings.merge(users, on='UserID').merge(movies, on='MovieID')
df['GenreList'] = df['Genres'].str.split('|')
df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s')
print(f"Merged: {df.shape}")
df.head()

Merged: (1000209, 12)


,UserID,MovieID,Rating,Timestamp,Gender,Age,Occupation,Zipcode,Title,Genres,GenreList,Datetime
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,[Drama],2000-12-31 22:12:40
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,"[Animation, Children's, Musical]",2000-12-31 22:35:09
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance,"[Musical, Romance]",2000-12-31 22:32:48
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama,[Drama],2000-12-31 22:04:35
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,"[Animation, Children's, Comedy]",2001-01-06 23:38:11


In [6]:
# movies['GenreList'] = movies['Genres'].str.split('|')
# movies['GenreList'] = movies['GenreList'].fillna("").astype('str')
movies['MovieID'].head()

,MovieID
0,1
1,2
2,3
3,4
4,5


# 2. Content-based Filtering Pipeline

In [7]:
user_counts  = df.groupby('UserID')['Rating'].count()
movie_counts = df.groupby('MovieID')['Rating'].count()
active_users  = user_counts[user_counts  >= 50].index
active_movies = movie_counts[movie_counts >= 50].index

df_filtered = df[df['UserID'].isin(active_users) & df['MovieID'].isin(active_movies)].copy()
print(f"Filtered: {df_filtered['UserID'].nunique():,} users | "
      f"{df_filtered['MovieID'].nunique():,} movies | "
      f"{len(df_filtered):,} ratings")

Filtered: 4,297 users | 2,514 movies | 922,127 ratings


In [8]:
train_df, test_df = train_test_split(df_filtered, test_size=0.2, random_state=42, stratify=df_filtered['UserID'])
print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")

Train: 737,701  |  Test: 184,426


In [9]:
# Lấy danh sách toàn bộ movie có trong df_filtered
movies_profile = df_filtered[['MovieID', 'Title', 'Genres']].drop_duplicates('MovieID').sort_values('MovieID').reset_index(drop=True)

# Khởi tạo và fit TF-IDF
tf_v2 = TfidfVectorizer(tokenizer=lambda x: x.split('|'), token_pattern=None)
tfidf_matrix = tf_v2.fit_transform(movies_profile['Genres'])

# Dictionary để tra cứu nhanh: Nhập MovieID -> Trả ra vị trí dòng trong tfidf_matrix
movie_idx_map = pd.Series(movies_profile.index, index=movies_profile['MovieID']).to_dict()

print(f"Kích thước TF-IDF Matrix: {tfidf_matrix.shape}")

Kích thước TF-IDF Matrix: (2514, 18)


In [10]:
# import plotly.express as px

# # 1. Tính toán ma trận tương đồng Cosine cho toàn bộ tfidf_matrix
# # Kích thước ma trận sẽ là 2514x2514
# cosine_sim_all = cosine_similarity(tfidf_matrix[:100], tfidf_matrix[:100])

# # 2. Lấy danh sách MovieID để làm nhãn trục
# movie_ids = movies_profile['MovieID'].iloc[:100].values

# # 3. Vẽ Heatmap tổng quát
# fig = px.imshow(
#     cosine_sim_all,
#     x=movie_ids,
#     y=movie_ids,
#     color_continuous_scale='Viridis',
#     labels=dict(x="MovieID", y="MovieID", color="Cosine Similarity")
# )

# fig.update_layout(
#     width=1000,
#     height=900
# )

# # Tắt hiển thị tick labels nếu quá dày đặc để nhìn rõ cấu trúc khối
# fig.update_xaxes(showticklabels=True, tickangle=-45)
# fig.update_yaxes(showticklabels=True)

# fig.show()

In [11]:
# 1. Tạo ma trận thưa User-Item (Chỉ lấy thông tin từ tập Train)
train_pivot = train_df.pivot(index='UserID', columns='MovieID', values='Rating')

# 2. Mean-centering (Chuẩn hóa: Trừ đi rating trung bình của mỗi User)
user_means = train_pivot.mean(axis=1)
train_pivot_centered = train_pivot.sub(user_means, axis=0).fillna(0)

# 3. Đảm bảo cột Movie của pivot khớp chuẩn với thứ tự của tfidf_matrix
valid_movies = movies_profile['MovieID'].values
train_pivot_centered = train_pivot_centered.reindex(columns=valid_movies, fill_value=0)

# 4. Tính User Profile Vectors (V_u = Lịch sử Rating * Đặc trưng Phim)
user_item_sparse = csr_matrix(train_pivot_centered.values)
user_profiles = user_item_sparse.dot(tfidf_matrix) # Phép nhân này cực kỳ nhanh

# 5. Lưu thành DataFrame để dễ lookup
user_profiles_df = pd.DataFrame(user_profiles.toarray(), index=train_pivot_centered.index)
print(f"Kích thước User Profile Matrix: {user_profiles_df.shape}")

Kích thước User Profile Matrix: (4297, 18)


In [12]:
# 1. Lấy danh sách tên thể loại từ vectorizer
genre_names = tf_v2.get_feature_names_out()

# 2. Gán lại tên cột cho user_profiles_df
user_profiles_named = user_profiles_df.copy()
user_profiles_named.columns = genre_names

# 3. In ra profile của UserID = 1
print("User Profile for UserID 1 (Genre Weights):")
user_profiles_named[user_profiles_named.index == 1]

User Profile for UserID 1 (Genre Weights):


,action,adventure,animation,children's,comedy,crime,documentary,drama,fantasy,film-noir,horror,musical,mystery,romance,sci-fi,thriller,war,western
UserID,,,,,,,,,,,,,,,,,,
1,0.630187,0.306209,-1.040877,-0.061724,-0.91492,-0.340252,0.0,2.841372,-0.530577,0.0,0.0,0.460528,0.0,-1.981478,0.42359,-0.761953,1.2687,0.0


In [13]:
def genre_recommendations(user_id=1, top_x=10):
    """
    Returns top-x movie recommendations for a specific user based on their profile.
    Returns: DataFrame with Title, Genres, and Similarity Score
    """
    if user_id not in user_profiles_df.index:
        return f"User ID {user_id} not found."

    # 1. Lấy vector đặc trưng của User (đã được tính ở bước trước)
    u_vec = user_profiles_df.loc[[user_id]].values # Shape (1, 18)

    # 2. Tính toán độ tương đồng giữa User Profile và toàn bộ Movie TF-IDF
    # tfidf_matrix shape: (2514, 18)
    scores = cosine_similarity(u_vec, tfidf_matrix).flatten()

    # 3. Tạo DataFrame kết quả
    recommendations = movies_profile.copy()
    recommendations['score'] = scores

    # 4. Loại bỏ các phim user đã xem trong tập Train để tránh gợi ý lại
    seen_movies = train_df[train_df['UserID'] == user_id]['MovieID'].values
    recommendations = recommendations[~recommendations['MovieID'].isin(seen_movies)]

    # 5. Sắp xếp và lấy top x
    top_recommendations = recommendations.sort_values(by='score', ascending=False).head(top_x)

    return top_recommendations[['Title', 'Genres', 'score']]

# Chạy thử nghiệm cho UserID = 1
print("Top recommendations for User 1:")
genre_recommendations(user_id=1, top_x=20)

Top recommendations for User 1:


,Title,Genres,score
2492,Remember the Titans (2000),Drama,0.681288
13,Nixon (1995),Drama,0.681288
2491,Girlfight (2000),Drama,0.681288
2512,Tigerland (2000),Drama,0.681288
2511,Requiem for a Dream (2000),Drama,0.681288
875,"Crucible, The (1996)",Drama,0.681288
904,Ghosts of Mississippi (1996),Drama,0.681288
903,Marvin's Room (1996),Drama,0.681288
901,Bastard Out of Carolina (1996),Drama,0.681288
2027,"Cradle Will Rock, The (1999)",Drama,0.681288


# 3. Evaluation

In [15]:
RELEVANCE_THRESHOLD = 4.0   # ratings ≥ 4 considered "relevant"
K_VALUES = [5, 10, 20]
all_movies = movies_profile['MovieID'].tolist()

def dcg(scores):
    return sum(s / np.log2(i + 2) for i, s in enumerate(scores))

def ndcg_at_k(recommended, relevant_set, k):
    hits = [1 if m in relevant_set else 0 for m in recommended[:k]]
    ideal = sorted(hits, reverse=True)
    return dcg(hits) / dcg(ideal) if dcg(ideal) > 0 else 0.0

def ranking_metrics(test_data, k_values, n_users=200):
    """Vectorized version: compute all user-candidate scores in batch."""

    test_users  = test_data['UserID'].unique()
    train_users = set(train_df['UserID'].unique())
    eval_users  = list(set(test_users) & train_users)[:n_users]

    results = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in k_values}

    # ── 1. Build user matrix (only valid users) ──────────────────────────────
    valid_users = [u for u in eval_users if u in user_profiles_df.index]
    user_matrix = user_profiles_df.loc[valid_users].values  # (U, F)

    # ── 2. Build candidate matrix per user & score in bulk ───────────────────
    # Precompute: seen movies per user
    seen_per_user = (
        train_df.groupby('UserID')['MovieID']
        .apply(set)
        .to_dict()
    )

    # Global candidate pool = all movies (masking happens per-user below)
    all_movie_ids  = np.array(all_movies)                      # (M,)
    movie_indices  = np.array([movie_idx_map[m] for m in all_movies])
    all_movie_vecs = tfidf_matrix[movie_indices]               # (M, F) sparse

    # ── 3. Score matrix: (U, M) = cosine_similarity(user_matrix, movie_vecs) ─
    score_matrix = cosine_similarity(user_matrix, all_movie_vecs)  # (U, M)

    # ── 4. Per-user masking → ranking → metrics ──────────────────────────────
    # Relevant movies per user (from test set, vectorized via groupby)
    rel_dict = (
        test_data[test_data['Rating'] >= RELEVANCE_THRESHOLD]
        .groupby('UserID')['MovieID']
        .apply(set)
        .to_dict()
    )

    movie_id_to_col = {mid: i for i, mid in enumerate(all_movies)}  # M lookup

    for i, user_id in enumerate(valid_users):
        relevant = rel_dict.get(user_id, set())
        if not relevant:
            continue

        seen = seen_per_user.get(user_id, set())

        # Mask seen movies → set score to -inf
        user_scores = score_matrix[i].copy()                   # (M,)
        seen_cols   = [movie_id_to_col[m] for m in seen if m in movie_id_to_col]
        if seen_cols:
            user_scores[seen_cols] = -np.inf

        # Rank: argsort descending
        ranked_indices = np.argsort(user_scores)[::-1]
        ranked_movies  = all_movie_ids[ranked_indices]         # sorted MovieIDs

        # Metrics
        rel_array = np.isin(ranked_movies, list(relevant)).astype(int)  # (M,)

        for k in k_values:
            top_k_hits = rel_array[:k].sum()
            results[k]['precision'].append(top_k_hits / k)
            results[k]['recall'].append(top_k_hits / len(relevant))

            # NDCG
            hits_k  = rel_array[:k]
            ideal_k = np.sort(hits_k)[::-1]
            disc     = 1.0 / np.log2(np.arange(1, k + 1) + 1)  # (k,)
            dcg_val  = hits_k  @ disc
            idcg_val = ideal_k @ disc
            results[k]['ndcg'].append(dcg_val / idcg_val if idcg_val > 0 else 0.0)

    return {str(k): {m: np.mean(v) for m, v in metrics.items()} for k, metrics in results.items()}


# ── Usage ────────────────────────────────────────────────────────────────────
print("Computing ranking metrics (Vectorized Content-Based CF)...")
cbcf_rank = ranking_metrics(test_df, K_VALUES, n_users=150)
print(f"  P@10: {cbcf_rank['10']['precision']:.4f} | R@10: {cbcf_rank['10']['recall']:.4f} | NDCG@10: {cbcf_rank['10']['ndcg']:.4f}")

Computing ranking metrics (Vectorized Content-Based CF)...
  P@10: 0.0240 | R@10: 0.0137 | NDCG@10: 0.1051


In [16]:
print(f"\n{'Model':<16}", end='')
for k in K_VALUES:
    print(f"  P@{k}   R@{k}  NDCG@{k}", end='')
print()
print("-" * 80)

for name, metrics in [("Content-based", cbcf_rank)]:
    print(f"{name:<16}", end='')
    for k in K_VALUES:
        m = metrics[f'{k}']
        print(f"  {m['precision']:.4f} {m['recall']:.4f} {m['ndcg']:.4f}", end='')
    print()


Model             P@5   R@5  NDCG@5  P@10   R@10  NDCG@10  P@20   R@20  NDCG@20
--------------------------------------------------------------------------------
Content-based     0.0253 0.0094 0.0754  0.0240 0.0137 0.1051  0.0233 0.0250 0.1344


In [17]:
# ── (b) Precision@K, Recall@K, NDCG@K line charts ────────────────────────────
metrics_data = {
    "Precision@K": {name: [m[f'{k}']['precision'] for k in K_VALUES]
                    for name, m in [("Content-based", cbcf_rank)]},
    "Recall@K":    {name: [m[f'{k}']['recall']    for k in K_VALUES]
                    for name, m in [("Content-based", cbcf_rank)]},
    "NDCG@K":      {name: [m[f'{k}']['ndcg']      for k in K_VALUES]
                    for name, m in [("Content-based", cbcf_rank)]},
}

fig_rank = make_subplots(rows=1, cols=3,
                          subplot_titles=list(metrics_data.keys()))
colors = {'Content-based': 'steelblue'}

for col_i, (metric_name, model_dict) in enumerate(metrics_data.items(), start=1):
    for model_name, vals in model_dict.items():
        fig_rank.add_trace(
            go.Scatter(x=K_VALUES, y=vals, mode='lines+markers',
                       name=model_name, showlegend=(col_i == 1),
                       marker_color=colors[model_name]),
            row=1, col=col_i
        )
    fig_rank.update_xaxes(title_text='K', row=1, col=col_i)
    fig_rank.update_yaxes(title_text=metric_name, row=1, col=col_i)

fig_rank.update_layout(title='Ranking Metrics @K by Model',
                        template='plotly_white', height=400)
fig_rank.show()